In [ ]:
# Import Libraries
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

# 1. Load Dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Resize images to 224x224 (required for VGG16)
x_train = tf.image.resize(x_train, (224, 224)) / 255.0
x_test = tf.image.resize(x_test, (224, 224)) / 255.0

# Convert labels
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# 2. Load Pre-trained VGG16 Model (without top layer)
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224,224,3))

# 3. Freeze Base Layers
for layer in base_model.layers:
    layer.trainable = False

# 4. Add Custom Classifier Head
x = base_model.output
x = layers.Flatten()(x)
x = layers.Dense(128, activation='relu')(x)
output = layers.Dense(10, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)

# 5. Compile Model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# 6. Train (Frozen Model)
print("Training with Frozen Layers...")
history_frozen = model.fit(x_train, y_train,
                           epochs=5,
                           validation_data=(x_test, y_test))

# 7. Fine-Tuning (Unfreeze last few layers)
for layer in base_model.layers[-4:]:   # unfreeze last 4 layers
    layer.trainable = True

# Compile again with lower learning rate
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

print("Fine-tuning Model...")
history_finetune = model.fit(x_train, y_train,
                             epochs=5,
                             validation_data=(x_test, y_test))

# 8. Evaluate Model
test_loss, test_acc = model.evaluate(x_test, y_test)
print("\nFinal Test Accuracy:", test_acc)

In [ ]:
# Import Libraries
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Data Preprocessing (Using CIFAR-10)
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Resize images to 224x224 (required for VGG16)
x_train = tf.image.resize(x_train, (224, 224)) / 255.0
x_test = tf.image.resize(x_test, (224, 224)) / 255.0

# Convert labels to categorical
y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

# 2. Load Pre-trained VGG16 Model (without top layer)
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224,224,3))

# 3. Freeze Base Layers
for layer in base_model.layers:
    layer.trainable = False

# 4. Add Custom Classifier Head
model = models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

# 5. Compile Model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# 6. Train Only Top Layers (Frozen Base)
history_frozen = model.fit(x_train, y_train,
                           epochs=5,
                           validation_data=(x_test, y_test))

# 7. Fine-Tuning (Unfreeze last few layers)
for layer in base_model.layers[-4:]:   # unfreeze last 4 layers
    layer.trainable = True

# Recompile after unfreezing
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# 8. Train Again (Fine-tuned)
history_finetune = model.fit(x_train, y_train,
                             epochs=5,
                             validation_data=(x_test, y_test))

# 9. Evaluate Model
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Final Test Accuracy:", test_acc)